# Rover VLA — stage-one training on Colab

Fine-tunes **SmolVLA** (frozen VLM backbone, action expert trained) on the
rover waypoint-chunk dataset. Mirrors `rover/train_smoke.sh` but uses a Colab
GPU (A100/L4 = bf16 native; T4/V100 = fp32 fallback, auto-detected).

**Before running:** regenerate the dataset locally with the fixed sampler
(`rover/datagen/batch_datagen.sh` → `to_lerobot.py --chunk-k 10 --paraphrase`),
then upload it to Google Drive as a tarball — see the Dataset cell.

Pipeline: **data (local Gazebo) → train (Colab) → eval (local sim + policy_server).**
Colab does training only; datagen and closed-loop eval stay on the local rover host.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 2. Install
lerobot with the SmolVLA extra + wandb. Pin if Colab's base breaks the build.

In [ ]:
!pip install -q "lerobot[smolvla]" wandb
import lerobot, torch
print('lerobot', lerobot.__version__, '| torch', torch.__version__)

## 3. Dataset
Mount Drive and extract the LeRobot dataset tarball you uploaded.

Make the tarball locally after regenerating v3:
```bash
cd rover/data/lerobot && tar czf rover_vla_v3.tar.gz rover_vla_v3
# upload rover_vla_v3.tar.gz to Drive: MyDrive/rover_vla/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, tarfile
TAR = '/content/drive/MyDrive/rover_vla/rover_vla_v3.tar.gz'
DATA_ROOT = '/content/data'
os.makedirs(DATA_ROOT, exist_ok=True)
with tarfile.open(TAR) as t:
    t.extractall(DATA_ROOT)
DATASET_ROOT = f'{DATA_ROOT}/rover_vla_v3'
print('dataset at', DATASET_ROOT, '| exists:', os.path.isdir(DATASET_ROOT))

## 4. Weights & Biases
Put your key in a Colab **secret** named `WANDB_API_KEY` (key icon, left sidebar),
or paste it when prompted. Metrics stream to project `rover-vla`.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    import wandb; wandb.login()
print('wandb key set:', bool(os.environ.get('WANDB_API_KEY')))

## 5. Precision (auto)
bf16 needs compute capability ≥ 8.0 (A100/L4). On T4 (7.5) / V100 (7.0) patch
lerobot's hardcoded bf16 VLM load to fp32 — the same fix used for the Maxwell
Titan X. Also lifts VRAM, so batch size is set per GPU below.

In [ ]:
import torch, pathlib, re
cc = torch.cuda.get_device_capability()
bf16_ok = cc[0] >= 8
print('compute capability', cc, '| bf16 native:', bf16_ok)

if not bf16_ok:
    import lerobot
    f = pathlib.Path(lerobot.__file__).parent / 'policies/smolvla/smolvlm_with_expert.py'
    s = f.read_text().replace('torch_dtype="bfloat16"', 'torch_dtype="float32"')
    f.write_text(s)
    print('patched VLM load -> float32')

# batch by VRAM headroom (500M VLM + flow expert)
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 64 if gb > 30 else (32 if gb > 20 else 16)
print('total VRAM %.0f GB -> batch_size %d' % (gb, BATCH))

## 6. Train
Stage one: frozen backbone, action expert on the flat 10×(x,y,v)=30-dim chunk
(`chunk_size=1` — lerobot's delta-timestamps chunking would mix body frames).
10k steps ≈ 1 epoch of ~568 episodes. Checkpoints every 2k.

In [ ]:
OUTPUT_DIR = '/content/outputs/stage1_v3'
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --policy.push_to_hub=false \
  --policy.chunk_size=1 --policy.n_action_steps=1 \
  --dataset.repo_id=local/rover_vla_v3 \
  --dataset.root={DATASET_ROOT} \
  --dataset.video_backend=pyav \
  --rename_map='{{"observation.image": "observation.images.camera1"}}' \
  --batch_size={BATCH} --steps=10000 --save_freq=2000 --log_freq=50 \
  --wandb.enable=true --wandb.project=rover-vla --wandb.mode=online \
  --output_dir={OUTPUT_DIR}

## 7. Save the checkpoint back to Drive
Download `stage1_v3.tar.gz` from Drive to the local host, then eval with
`rover/runtime/policy_server.py` + `ros2 run rover_expert run_eval.py --swap`.

In [ ]:
import tarfile, os
os.makedirs('/content/drive/MyDrive/rover_vla/checkpoints', exist_ok=True)
OUT = '/content/drive/MyDrive/rover_vla/checkpoints/stage1_v3.tar.gz'
with tarfile.open(OUT, 'w:gz') as t:
    t.add(f'{OUTPUT_DIR}/checkpoints/last', arcname='stage1_v3_last')
print('saved', OUT)